<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing TEMPO NRT (Level 2)**



 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> Generate a Bearer Token.
4. <font size="3"> Get `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data



In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.net import create_session
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**
    
    
We are interested in `TEMPO NO2 tropospheric and stratospheric columns V04` data. This collection provides hourly data for level 2 data, considered Near Real Time (NRT).


In [ ]:
TEMPO_L2_NRTNO2_ccid = "C3685896872-LARC_CLOUD" # 
time_range = [dt.datetime(2025, 10, 1), dt.datetime(2025, 10, 7)] # One month of data

bounding_box = [-124.63309,46.35932,  -121, 49.83307] # WSEN area within Seattle PNW

cmr_urls = get_cmr_urls(ccid=TEMPO_L2_NRTNO2_ccid, bounding_box=bounding_box, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[0]


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()

# Accessing Metadata

<font size="3.5"> What are some tools, their differences, and what do they do.

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **PYDAP: Metadata-Only**



```{note}
Q: How do we tell the server which protocol to use?
A: By replaing the http -> dap4 in the URL
```


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

### Subset by Variables

We will retain `xtrack`, and all data within Groups `product` and `geolocation`


In [ ]:
output_path = "data/"

keep_variables = ["/xtrack","/mirror_step", 
                  "/product", "/geolocation/time",
                  "/geolocation/longitude","/geolocation/latitude", 
                 ]


# Stream data

each remote file is stored into an individual file. No data aggregation


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, keep_variables=keep_variables, output_path=output_path)

In [ ]:
%%time
dt = xr.open_datatree(output_path+"TEMPO_NO2_L2_V04_20251001T141426Z_S004G08.nc4")
dt['geolocation'].load()

In [ ]:
dt